# DVF 2022 — Compléments : nature de mutation, départements, natures de culture

Ce notebook calcule quatre relevés du fichier DVF 2022 : la distribution des natures de mutation, le nombre de départements présents, et le nombre de codes distincts des colonnes Nature culture et Nature culture spéciale. Ces valeurs complètent le tableau descriptif des 43 colonnes.

## Cellule 1 — Connexion au fichier

Interrogation avec DuckDB, directement au format Parquet. Le chemin pointe vers le fichier `dvf-2022.parquet` placé dans le dossier `data/`.

In [1]:
import duckdb
from pathlib import Path

# Chemin vers le fichier Parquet, place dans le dossier data/
FICHIER = Path(r"./data/dvf-2022.parquet")

con = duckdb.connect()
pq = str(FICHIER)
assert FICHIER.exists(), f"Fichier introuvable : {FICHIER}"
nb_lignes = con.execute(f"SELECT count(*) FROM '{pq}'").fetchone()[0]
print(f"Fichier : {FICHIER.name}")
print(f"Lignes  : {nb_lignes:,}".replace(',', ' '))

Fichier : dvf-2022.parquet
Lignes  : 4 617 590


## Cellule 2 — Distribution des natures de mutation

Nombre de lignes et part (%) pour chaque valeur de la colonne *Nature mutation*, classées de la plus fréquente à la moins fréquente.

In [2]:
nature_mutation = con.execute(f"""
    SELECT "Nature mutation" AS nature,
           count(*) AS nb_lignes,
           ROUND(100.0 * count(*) / SUM(count(*)) OVER (), 1) AS part_pct
    FROM '{pq}'
    GROUP BY "Nature mutation"
    ORDER BY nb_lignes DESC
""").fetchdf()

print(f"Nombre de modalites distinctes : {len(nature_mutation)}")
nature_mutation

Nombre de modalites distinctes : 6


,nature,nb_lignes,part_pct
0,Vente,4267222,92.4
1,Vente en l'état futur d'achèvement,280574,6.1
2,Echange,45200,1.0
3,Vente terrain à bâtir,14268,0.3
4,Adjudication,9424,0.2
5,Expropriation,902,0.0


## Cellule 3 — Nombre de départements présents

Nombre de valeurs distinctes de la colonne *Code département*.

In [3]:
nb_departements = con.execute(f'SELECT count(DISTINCT "Code departement") FROM \'{pq}\'').fetchone()[0]
print(f"Departements distincts presents dans le fichier : {nb_departements}")

# Liste des codes departement presents (pour controle)
departements = con.execute(f'''
    SELECT DISTINCT "Code departement" AS code
    FROM '{pq}'
    ORDER BY code
''').fetchdf()
departements

Departements distincts presents dans le fichier : 97


,code
0,01
1,02
2,03
3,04
4,05
...,...
92,95
93,971
94,972
95,973


## Cellule 4 — Codes distincts des natures de culture

Nombre de codes distincts réellement présents dans le fichier pour les colonnes *Nature culture* et *Nature culture spéciale* (les valeurs vides ne sont pas comptées).

In [4]:
res = con.execute(f'''
    SELECT count(DISTINCT "Nature culture") AS nature_culture,
           count(DISTINCT "Nature culture speciale") AS nature_culture_speciale
    FROM '{pq}'
''').fetchone()
print(f"Codes distincts, Nature culture          : {res[0]}")
print(f"Codes distincts, Nature culture speciale : {res[1]}")

Codes distincts, Nature culture          : 27
Codes distincts, Nature culture speciale : 127
